# Chapter 12: Prompt Engineering & In-Context Learning
## Notebook 01 — Prompt Basics

This notebook introduces the core building blocks of prompt engineering: **prompt anatomy**, **zero-shot vs few-shot** prompting, **system vs user** roles, **structured outputs** with Pydantic, and the **sensitivity** of LLM responses to wording.

### What you'll learn

| Topic | Section |
|-------|--------|
| Prompt anatomy (instruction, context, input, output spec) | §2 |
| Zero-shot, few-shot, and in-context learning | §3 |
| System vs user prompts | §4 |
| Structured outputs with Pydantic schemas | §5 |
| Sensitivity to wording | §6 |
| The mock LLM client (offline) | §7 |

**Estimated time:** 1.5–2 hours

---
*Generated by Berta AI*

---
## 1. Introduction & Setup

We use **jinja2** for prompt templating, **pydantic** for output schemas, and a deterministic **MockLLMClient** so this notebook runs **fully offline**. To call a real model later, install `openai` or `anthropic` and swap the client — the rest of the code is unchanged.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))

from prompt_templates import (
    PromptTemplate, FewShotTemplate, FewShotExample,
    ChainOfThoughtTemplate, default_registry,
)
from llm_clients import MockLLMClient, EchoLLMClient
from evaluation_utils import safe_json_parse

import json
import textwrap

client = MockLLMClient(model='mock-llm-v1', temperature=0.0)
echo = EchoLLMClient()
print('Setup complete. Default model:', client.model)

---
## 2. Prompt Anatomy

Every effective prompt has four ingredients:

1. **Instruction** — what to do ("classify the sentiment").
2. **Context** — background or examples the model can lean on.
3. **Input** — the new data to act on.
4. **Output spec** — the format the answer must take ("return only the label").

Let's build a minimal one and inspect what we send.

In [ ]:
tmpl = PromptTemplate(
    name='sentiment_v1',
    system='You are a careful sentiment classifier.',
    template=("""Classify the sentiment as 'positive', 'negative', or 'neutral'.
Return only the label.

Text: {{ text }}
Label:"""),
)

rendered = tmpl.render(text='I absolutely loved this movie!')
print('--- Rendered prompt ---')
print(rendered)
print('\n--- Messages (chat-style) ---')
for m in tmpl.render_messages(text='I absolutely loved this movie!'):
    print(f"[{m['role']}] {m['content'][:80]}...")
print('\nFingerprint:', tmpl.fingerprint())

---
## 3. Zero-Shot, Few-Shot & In-Context Learning

- **Zero-shot**: instruction only. Cheapest, but quality depends on the model knowing the task.
- **Few-shot**: a handful of labeled examples in the prompt. The model **learns from context** without any weight updates — this is **in-context learning**.
- **Trade-off**: more examples → better grounding but more tokens (cost + latency).

In [ ]:
# Zero-shot
zero = PromptTemplate(
    name='topic_zero_shot',
    template='Classify the topic of the headline as one of: tech, sports, politics.\n\nHeadline: {{ headline }}\nTopic:',
)
print('Zero-shot prompt:')
print(zero.render(headline='Team wins championship in overtime'))
print()

# Few-shot
few = FewShotTemplate(
    name='topic_few_shot',
    template='',
    examples=[
        FewShotExample('New phone announced today.', 'tech'),
        FewShotExample('Election results came in late.', 'politics'),
        FewShotExample('Star quarterback returns from injury.', 'sports'),
    ],
)
print('Few-shot prompt:')
print(few.render(input='Team wins championship in overtime', instruction='Classify the topic.'))

In [ ]:
# Sending both through the mock client
for label, prompt in [
    ('zero', zero.render(headline='Team wins championship in overtime')),
    ('few', few.render(input='Team wins championship in overtime', instruction='Classify the topic.')),
]:
    response = client.complete(prompt)
    print(f'[{label}] -> {response.text!r}  (tokens: {response.total_tokens})')

---
## 4. System vs User Prompts

Modern chat APIs separate **system** prompts (persona, rules, output spec) from **user** prompts (the request itself). The split helps because:

- The system message can be cached/reused across requests.
- Defenses against injection rely on the model trusting the system prompt **more** than user-supplied text.
- It separates **policy** from **data**.

In [ ]:
msgs = tmpl.render_messages(text='Service was awful and the room was dirty.')
for m in msgs:
    print(f"[{m['role']:6}] {m['content']}")
print('\nMock LLM chat() response:', client.chat(msgs).text)

---
## 5. Structured Outputs with Pydantic

Free-form text is hard to parse. Asking the model to return **JSON conforming to a schema** — and then validating with Pydantic — makes downstream code reliable. We define the schema once and use it for **both** the prompt instructions and the parser.

In [ ]:
from pydantic import BaseModel, Field, ValidationError
from typing import Literal

class SentimentResult(BaseModel):
    label: Literal['positive', 'negative', 'neutral']
    confidence: float = Field(ge=0.0, le=1.0)
    snippet: str

# Build the prompt from the schema
schema_json = json.dumps(SentimentResult.model_json_schema(), indent=2)
print('Schema:')
print(schema_json[:300] + '...')

structured = PromptTemplate(
    name='sentiment_json',
    system='You are a strict JSON-only sentiment classifier.',
    template=(
        'Return a single JSON object matching this schema:\n'
        '```json\n' + schema_json + '\n```\n\n'
        'Text: {{ text }}\nJSON:'
    ),
)

raw = client.complete(structured.render(text='Loved it!')).text
print('\nRaw response:', raw)

parsed = safe_json_parse(raw)
print('Parsed dict:', parsed)
try:
    obj = SentimentResult.model_validate(parsed)
    print('Validated object:', obj)
except (ValidationError, TypeError) as e:
    print('Validation failed:', e)

**Tip.** Always pair a structured-output prompt with a **safe parser**. The bundled `safe_json_parse` first tries `json.loads`, then falls back to extracting the first `{...}` block. This handles the common case where the model wraps JSON in prose.

---
## 6. Sensitivity to Wording

LLMs are surprisingly sensitive to small phrasing changes. Below we send the same task through several wordings and compare outputs from the mock client. (Real LLMs show even larger swings — always check empirically.)

In [ ]:
text = 'The food was great but the service was awful.'

variants = [
    'Is this review positive or negative? Text: ' + text,
    'Sentiment of: ' + text,
    'Classify as positive, negative, or neutral. Text: ' + text + '\nLabel:',
    'Tell me how the customer feels: ' + text,
]

for v in variants:
    out = client.complete(v).text
    print(f'INPUT (first 60 chars): {v[:60]!r:62} -> {out!r}')

**Why this matters.** Different phrasings can flip the model's interpretation between *classification* and *free-form description*. In production:

- Pin a **single canonical wording** per task and version it.
- Test wording changes against your evaluation harness before rolling out.
- Prefer **explicit output specs** ("return only one of: positive, negative, neutral") over implicit ones.

---
## 7. The Mock LLM Client

This chapter ships with `MockLLMClient`, a deterministic rule-based stub that:

- Detects sentiment / extraction / math / CoT / ReAct prompts.
- Honors `temperature` and `seed` for reproducible variation (used in self-consistency demos in Notebook 02).
- Returns an `LLMResponse` with token counts so eval harnesses behave like the real thing.

This means notebooks run **without API keys, network, or cost** — yet exercise every prompt-engineering pattern.

In [ ]:
# Same prompt, two temperatures
prompt = 'Q: 2 plus 3 plus 4. Let\'s think step by step.'
for t, s in [(0.0, 0), (0.7, 1), (0.7, 2), (0.7, 3)]:
    r = client.complete(prompt, temperature=t, seed=s)
    print(f'temp={t} seed={s}: {r.text.strip()[:70]}')

---
## 8. Key Takeaways

- A prompt has **four parts**: instruction, context, input, output spec — design each deliberately.
- **Few-shot** beats zero-shot for narrow tasks; pay the token cost for stability.
- **System** prompts hold policy; **user** prompts hold data — keep them separate.
- Always pair **structured-output prompts** with **schema validation** and a **safe parser**.
- LLMs are **sensitive to wording**: pin one canonical phrasing and version it.

Next: **Notebook 02** — chain-of-thought, self-consistency, and ReAct.

---
*Generated by Berta AI*